In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/cleaned_data.csv")
print("Shape:", df.shape)
df.head()

Shape: (13200, 6)


,location,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2,3699.810606
1,Chikka Tirupathi,2600.0,5.0,120.00,4,4615.384615
2,Uttarahalli,1440.0,2.0,62.00,3,4305.555556
3,Lingadheeranahalli,1521.0,3.0,95.00,3,6245.890861
4,Kothanur,1200.0,2.0,51.00,2,4250.000000


In [ ]:
print("Unique locations:", df["location"].nunique())
df["location"].value_counts().head(10)

Unique locations: 1298


location
Whitefield               532
Sarjapur  Road           392
Electronic City          302
Kanakpura Road           264
Thanisandra              232
Yelahanka                210
Uttarahalli              186
Hebbal                   176
Marathahalli             175
Raja Rajeshwari Nagar    171
Name: count, dtype: int64

In [ ]:
location_stats = df["location"].value_counts()
rare_locations = location_stats[location_stats <= 10].index
print("Rare locations:", len(rare_locations))

df["location"] = df["location"].apply(lambda x: "other" if x in rare_locations else x)
print("Unique locations after grouping:", df["location"].nunique())

Rare locations: 1058
Unique locations after grouping: 241


In [ ]:
df = df[df["total_sqft"] / df["bhk"] >= 300]
print("Shape after removing bhk outliers:", df.shape)

Shape after removing bhk outliers: (12456, 6)


In [ ]:
def remove_pps_outliers(df):
    out = pd.DataFrame()
    for loc, subdf in df.groupby("location"):
        m = np.mean(subdf["price_per_sqft"])
        sd = np.std(subdf["price_per_sqft"])
        reduced = subdf[
            (subdf["price_per_sqft"] > (m - sd)) & (subdf["price_per_sqft"] <= (m + sd))
        ]
        out = pd.concat([out, reduced], ignore_index=True)
    return out


df = remove_pps_outliers(df)
print("Shape after pps outliers:", df.shape)

Shape after pps outliers: (10245, 6)


In [ ]:
df = df[df["bath"] < df["bhk"] + 2]
print("Shape after bath filter:", df.shape)

Shape after bath filter: (10151, 6)


In [9]:
df.to_csv("../data/engineered_data.csv", index=False)
print("Saved engineered_data.csv — rows:", len(df))

Saved engineered_data.csv — rows: 10151
